# 工業 / 半導體 YOLO26 檢測（Kaggle）

1. 右側 **Settings**：Accelerator 選 **GPU**，**Internet** 打開。
2. 用 **Add Data** 加入程式（本 repo 的 `kaggle/` 資料夾）與資料集，例如 **PCB Defects** 或 **WM-811K**。
3. 依序執行下面的格子。輸出在 `/kaggle/working/semiconductor/`。

In [ ]:
# 取得程式碼：
#   A) 已把本 repo 的 kaggle/ 資料夾上傳成 Kaggle Dataset 並用 Add Data 加入 → 自動找到
#   B) 否則從 GitHub clone（repo 需為公開）
import glob, os, subprocess

hits = [p for d in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * d + "vi_core.py")]
if hits:
    CODE = os.path.dirname(hits[0])
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/MorrisLaLa/visual-inspection.git", "/kaggle/working/visual-inspection"],
                   check=True)
    CODE = "/kaggle/working/visual-inspection/kaggle"
print("程式位置：", CODE)
!pip install -q -U ultralytics

## PCB Defects（VOC XML，大圖小缺陷 → 自動切塊）

In [ ]:
DATA = "/kaggle/input/<pcb-defects>/PCB_DATASET"   # 改成 Add Data 後的實際路徑
!python {CODE}/semiconductor_yolo.py --data {DATA} --imgsz 640 --epochs 100

In [ ]:
# 顯示報告與訓練圖表
from pathlib import Path
from IPython.display import Image, Markdown, display

OUT = Path("/kaggle/working/semiconductor")
display(Markdown((OUT / "report.md").read_text()))
for f in ["train/results.png", "train/confusion_matrix_normalized.png", "val_test/BoxPR_curve.png"]:
    if (OUT / f).exists():
        display(Image(filename=str(OUT / f), width=900))

## 其他資料（取消註解執行）

In [ ]:
# WM-811K 晶圓圖分類（依 lotName 分組；未標註晶圓自動進複判佇列）
# !python {CODE}/semiconductor_yolo.py --data /kaggle/input/<wm811k>/LSWMD.pkl --wafer-max-per-class 3000 --epochs 50

# 自有 AOI / SEM 影像（YOLO 或 COCO），用檔名裡的批號分組
# !python {CODE}/semiconductor_yolo.py --data /kaggle/input/<my-aoi>/ --group-regex '^(LOT\w+)_'

## 閉環：下一輪
把 `review_queue.zip` 下載下來人工修正標註、併入資料集後，用上一輪的 `best.pt` 接續訓練並比較：

In [ ]:
# !python {CODE}/semiconductor_yolo.py --data <新版資料> --model /kaggle/input/<上一輪輸出>/best.pt --baseline /kaggle/input/<上一輪輸出>/best.pt --unlabeled <未標註影像資料夾>